# Data Quality Validation

This notebook validates the Silver-layer Michelin restaurant dataset before it is used for analytics.

Quality checks cover:
- Row-count reconciliation
- Restaurant ID uniqueness
- Required-field completeness
- Valid price tiers
- Valid Michelin star values
- Geographic coordinate validity
- Business-rule consistency

### 1. Bronze → Silver reconciliation

In [0]:
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT restaurant_id) AS unique_ids,
    SUM(CASE WHEN restaurant_id IS NULL THEN 1 ELSE 0 END) AS null_ids
FROM workspace.michelin_analytics.silver_michelin_restaurants;

total_rows,unique_ids,null_ids
19622,19622,0


### 2. Primary-key quality

In [0]:
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT restaurant_id) AS unique_ids,
    SUM(CASE WHEN restaurant_id IS NULL THEN 1 ELSE 0 END) AS null_ids
FROM workspace.michelin_analytics.silver_michelin_restaurants;

total_rows,unique_ids,null_ids
19622,19622,0


### 3. Required fields

In [0]:
SELECT
    SUM(CASE WHEN restaurant_name IS NULL OR TRIM(restaurant_name) = '' THEN 1 ELSE 0 END) AS invalid_name,
    SUM(CASE WHEN address IS NULL OR TRIM(address) = '' THEN 1 ELSE 0 END) AS invalid_address,
    SUM(CASE WHEN location IS NULL OR TRIM(location) = '' THEN 1 ELSE 0 END) AS invalid_location,
    SUM(CASE WHEN cuisine IS NULL OR TRIM(cuisine) = '' THEN 1 ELSE 0 END) AS invalid_cuisine,
    SUM(CASE WHEN award IS NULL OR TRIM(award) = '' THEN 1 ELSE 0 END) AS invalid_award
FROM workspace.michelin_analytics.silver_michelin_restaurants;

invalid_name,invalid_address,invalid_location,invalid_cuisine,invalid_award
0,0,0,0,0


### 4. Business-rule validation

In [0]:
SELECT
    SUM(CASE
        WHEN price_tier IS NOT NULL
         AND price_tier NOT BETWEEN 1 AND 4
        THEN 1 ELSE 0
    END) AS invalid_price_tier,

    SUM(CASE
        WHEN star_count NOT BETWEEN 0 AND 3
        THEN 1 ELSE 0
    END) AS invalid_star_count,

    SUM(CASE
        WHEN is_starred = TRUE AND star_count = 0
        THEN 1 ELSE 0
    END) AS inconsistent_star_flag,

    SUM(CASE
        WHEN is_starred = FALSE AND star_count > 0
        THEN 1 ELSE 0
    END) AS inconsistent_nonstar_flag,

    SUM(CASE
        WHEN latitude NOT BETWEEN -90 AND 90
        THEN 1 ELSE 0
    END) AS invalid_latitude,

    SUM(CASE
        WHEN longitude NOT BETWEEN -180 AND 180
        THEN 1 ELSE 0
    END) AS invalid_longitude

FROM workspace.michelin_analytics.silver_michelin_restaurants;

invalid_price_tier,invalid_star_count,inconsistent_star_flag,inconsistent_nonstar_flag,invalid_latitude,invalid_longitude
0,0,0,0,0,0
